In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSVs (they are in current working directory)
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')
df_log = pd.read_csv('log.csv')
df_err = pd.read_csv('error_logs.csv')

# Ensure numeric for 'value' where present
if 'value' in df_metric.columns:
    df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
if 'value' in df_trace.columns:
    df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')
if 'value' in df_log.columns:
    df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')

# helper to convert min/max timestamp to Asia/Shanghai tz string
def ts_range_info(series):
    if series.size == 0:
        return None, None
    smin = pd.to_datetime(series.min(), unit='s', utc=True).tz_convert(tz)
    smax = pd.to_datetime(series.max(), unit='s', utc=True).tz_convert(tz)
    return smin.strftime('%Y-%m-%d %H:%M:%S %Z'), smax.strftime('%Y-%m-%d %H:%M:%S %Z')

# 1) File summaries: total rows, min/max timestamp, up to 20 unique cmdb_id sorted alphabetically
def file_summary(df, fname):
    rows = len(df)
    if 'timestamp' in df.columns:
        min_ts, max_ts = ts_range_info(df['timestamp'])
    else:
        min_ts = max_ts = None
    if 'cmdb_id' in df.columns:
        cmdbs = sorted(pd.unique(df['cmdb_id'].dropna().astype(str)))
        sample_cmdb = cmdbs[:20]
    else:
        sample_cmdb = []
    return pd.DataFrame([{
        'file': fname,
        'rows': rows,
        'min_ts(Asia/Shanghai)': min_ts,
        'max_ts(Asia/Shanghai)': max_ts,
        'sample_cmdb_up_to_20': ','.join(sample_cmdb)
    }])

metric_summary = file_summary(df_metric, 'metric.csv')
trace_summary = file_summary(df_trace, 'trace.csv')
log_summary = file_summary(df_log, 'log.csv')
errorlog_summary = file_summary(df_err, 'error_logs.csv')

# 2) metric.csv: up to 50 unique kpi_name values, and global P95 of 'value' per (cmdb_id,kpi_name)
# List up to 50 unique kpi_name
unique_kpis = sorted(pd.unique(df_metric['kpi_name'].dropna().astype(str)))[:50]
metric_kpi_list = pd.DataFrame({'kpi_name_sample_up_to_50': unique_kpis})

# Compute global P95 per (cmdb_id, kpi_name) using entire series
# Groupby quantile(0.95)
if {'cmdb_id','kpi_name','value'}.issubset(df_metric.columns):
    metric_p95 = df_metric.groupby(['cmdb_id','kpi_name'], dropna=False)['value'] \
                          .quantile(0.95) \
                          .reset_index(name='p95') \
                          .sort_values('p95', ascending=False)
    metric_p95['p95'] = metric_p95['p95'].round(6)
    metric_p95_top20 = metric_p95.head(20).reset_index(drop=True)
else:
    metric_p95_top20 = pd.DataFrame(columns=['cmdb_id','kpi_name','p95'])

# 3) trace.csv: up to 50 unique trace_name values, compute global P95 per (cmdb_id,trace_name)
unique_traces = sorted(pd.unique(df_trace['trace_name'].dropna().astype(str)))[:50]
trace_name_list = pd.DataFrame({'trace_name_sample_up_to_50': unique_traces})

if {'cmdb_id','trace_name','value'}.issubset(df_trace.columns):
    trace_p95 = df_trace.groupby(['cmdb_id','trace_name'], dropna=False)['value'] \
                        .quantile(0.95) \
                        .reset_index(name='p95') \
                        .sort_values('p95', ascending=False)
    trace_p95['p95'] = trace_p95['p95'].round(6)
    trace_p95_top20 = trace_p95.head(20).reset_index(drop=True)
else:
    trace_p95_top20 = pd.DataFrame(columns=['cmdb_id','trace_name','p95'])

# 4) log.csv: up to 50 unique log_name values
unique_logs = sorted(pd.unique(df_log['log_name'].dropna().astype(str)))[:50]
log_name_list = pd.DataFrame({'log_name_sample_up_to_50': unique_logs})

# 5) error_logs.csv: up to 5 sample rows (timestamp,cmdb_id,message) and count of distinct messages
if {'timestamp','cmdb_id','message'}.issubset(df_err.columns):
    # create a small sample with converted timestamp
    sample_err = df_err[['timestamp','cmdb_id','message']].copy().head(5)
    sample_err['timestamp'] = pd.to_datetime(sample_err['timestamp'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    distinct_message_count = int(df_err['message'].nunique(dropna=True))
else:
    sample_err = pd.DataFrame(columns=['timestamp','cmdb_id','message'])
    distinct_message_count = 0

# Compact outputs: summaries and top lists
metric_summary, trace_summary, log_summary, errorlog_summary, metric_kpi_list, metric_p95_top20, trace_name_list, trace_p95_top20, log_name_list, sample_err, distinct_message_count

```
Out[1]:
```
# Build a plain-English summary string from previously computed variables
# Reuse variables from prior execution: metric_summary, trace_summary, log_summary, errorlog_summary,
# metric_kpi_list, metric_p95_top20, trace_name_list, trace_p95_top20, log_name_list, sample_err, distinct_message_count

# Helper to get sample cmdb list from the summary cell (comma-joined)
def get_sample_cmdb(df_summary):
    s = df_summary.loc[0, 'sample_cmdb_up_to_20']
    if pd.isna(s) or s == '':
        return '[]'
    parts = [p.strip() for p in s.split(',') if p.strip()]
    return '[' + ', '.join(parts[:10]) + (' ...' if len(parts) > 10 else '') + ']'

# Extract basic file summaries
m_rows = int(metric_summary.loc[0,'rows'])
m_min = metric_summary.loc[0,'min_ts(Asia/Shanghai)']
m_max = metric_summary.loc[0,'max_ts(Asia/Shanghai)']
m_cmdb_sample = get_sample_cmdb(metric_summary)

t_rows = int(trace_summary.loc[0,'rows'])
t_min = trace_summary.loc[0,'min_ts(Asia/Shanghai)']
t_max = trace_summary.loc[0,'max_ts(Asia/Shanghai)']
t_cmdb_sample = get_sample_cmdb(trace_summary)

l_rows = int(log_summary.loc[0,'rows'])
l_min = log_summary.loc[0,'min_ts(Asia/Shanghai)']
l_max = log_summary.loc[0,'max_ts(Asia/Shanghai)']
l_cmdb_sample = get_sample_cmdb(log_summary)

e_rows = int(errorlog_summary.loc[0,'rows'])
e_min = errorlog_summary.loc[0,'min_ts(Asia/Shanghai)']
e_max = errorlog_summary.loc[0,'max_ts(Asia/Shanghai)']
e_cmdb_sample = get_sample_cmdb(errorlog_summary)

# Counts and samples
metric_kpi_count = int(df_metric['kpi_name'].nunique(dropna=True)) if 'kpi_name' in df_metric.columns else 0
metric_kpis_sample = list(metric_kpi_list['kpi_name_sample_up_to_50'].head(6)) if not metric_kpi_list.empty else []

trace_name_count = int(df_trace['trace_name'].nunique(dropna=True)) if 'trace_name' in df_trace.columns else 0
trace_names_sample = list(trace_name_list['trace_name_sample_up_to_50'].head(6)) if not trace_name_list.empty else []

log_name_count = int(df_log['log_name'].nunique(dropna=True)) if 'log_name' in df_log.columns else 0
log_names_sample = list(log_name_list['log_name_sample_up_to_50'].head(6)) if not log_name_list.empty else []

# Top examples (take up to 3)
def top_examples(df_top, cols, n=3):
    if df_top.shape[0] == 0:
        return []
    rows = []
    for _, r in df_top.head(n).iterrows():
        vals = tuple(r[c] for c in cols)
        rows.append(vals)
    return rows

metric_top3 = top_examples(metric_p95_top20, ['cmdb_id','kpi_name','p95'], 3)
trace_top3 = top_examples(trace_p95_top20, ['cmdb_id','trace_name','p95'], 3)

# Error log samples count of distinct messages
err_sample_count = sample_err.shape[0]
err_distinct_msgs = int(distinct_message_count)

# Compose summary string
summary = (
    "Telemetry summary (concise):\n\n"
    "1) Files:\n"
    f"- metric.csv: {m_rows} rows, time range {m_min} to {m_max}, sample cmdbs {m_cmdb_sample}\n"
    f"- trace.csv:  {t_rows} rows, time range {t_min} to {t_max}, sample cmdbs {t_cmdb_sample}\n"
    f"- log.csv:    {l_rows} rows, time range {l_min} to {l_max}, sample cmdbs {l_cmdb_sample}\n"
    f"- error_logs.csv: {e_rows} rows, time range {e_min} to {e_max}, sample cmdbs {e_cmdb_sample}\n\n"
    "2) metric.csv details:\n"
    f"- Unique KPI names: {metric_kpi_count} (showing up to 6 samples: {metric_kpis_sample})\n"
    "- Top P95 by (cmdb_id, kpi_name, P95) — highest 3 examples:\n"
)

if metric_top3:
    for cmdb,kpi,p95 in metric_top3:
        summary += f"  • {cmdb} | {kpi} | P95={p95}\n"
else:
    summary += "  (no metric P95 data)\n"

summary += (
    "\n3) trace.csv details:\n"
    f"- Unique trace names: {trace_name_count} (showing up to 6 samples: {trace_names_sample})\n"
    "- Top P95 by (cmdb_id, trace_name, P95) — highest 3 examples:\n"
)

if trace_top3:
    for cmdb,name,p95 in trace_top3:
        summary += f"  • {cmdb} | {name} | P95={p95}\n"
else:
    summary += "  (no trace P95 data)\n"

summary += (
    "\n4) log.csv details:\n"
    f"- Unique log_name values: {log_name_count} (samples up to 6: {log_names_sample})\n\n"
    "5) error_logs.csv details:\n"
    f"- Showing up to {err_sample_count} sample rows (timestamps shown earlier). Distinct message count: {err_distinct_msgs}.\n\n"
    "Notes: Full top-20 lists and full tables are available in the variables metric_p95_top20, trace_p95_top20,\n"
    "and the full dataframes df_metric, df_trace, df_log, df_err if you want more detail."
)

# Display the summary string as the final result
summary

The original code execution output of IPython Kernel is also provided below for reference:

(         file    rows    min_ts(Asia/Shanghai)    max_ts(Asia/Shanghai)                               sample_cmdb_up_to_20
0  metric.csv  151992  2022-03-21 20:00:00 CST  2022-03-21 20:29:00 CST  adservice,adservice-0,adservice-1,adservice-2,...,         file   rows    min_ts(Asia/Shanghai)    max_ts(Asia/Shanghai)                               sample_cmdb_up_to_20
0  trace.csv  36244  2022-03-21 20:00:00 CST  2022-03-21 20:29:00 CST  adservice-0,adservice-1,adservice-2,adservice2...,       file  rows    min_ts(Asia/Shanghai)    max_ts(Asia/Shanghai)                               sample_cmdb_up_to_20
0  log.csv  1814  2022-03-21 20:00:00 CST  2022-03-21 20:29:00 CST  adservice-0,adservice-1,adservice-2,cartservic...,              file  rows    min_ts(Asia/Shanghai)    max_ts(Asia/Shanghai)              sample_cmdb_up_to_20
0  error_logs.csv  1726  2022-03-21 20:00:00 CST  2022-03-21 20:29:58 CST  frontend-0,frontend-1,frontend-2,                              kpi_name_sample_up_to_50
0                                      app.grpc.count
1                                        app.grpc.mrt
2                                         app.grpc.rr
3                                         app.grpc.sr
4                                      app.http.count
..                                                ...
45             container.node-5.container_memory_swap
46         container.node-5.container_memory_usage_MB
47   container.node-5.container_memory_working_set_MB
48  container.node-5.container_network_receive_MB....
49  container.node-5.container_network_receive_err...

[50 rows x 1 columns],        cmdb_id                                           kpi_name           p95
0   adservice2                runtime.java_lang_Runtime_StartTime  1.647822e+12
1    adservice                runtime.java_lang_Runtime_StartTime  1.647523e+12
2    adservice   runtime.java_lang_OperatingSystem_ProcessCpuTime  4.102258e+11
3   adservice2   runtime.java_lang_OperatingSystem_ProcessCpuTime  2.037672e+11
4   adservice2   runtime.java_lang_Threading_CurrentThreadCpuTime  1.489348e+10
5   adservice2  runtime.java_lang_Threading_CurrentThreadUserTime  1.385067e+10
6   adservice2  runtime.java_lang_Threading_CurrentThreadAlloc...  1.046750e+10
7       node-4                                  system.disk.total  7.616300e+09
8       node-4                             system.fs.inodes.total  6.467040e+09
9       node-4                              system.fs.inodes.free  6.464446e+09
10      node-3                                  system.disk.total  5.867011e+09
11      node-2                                  system.disk.total  5.867011e+09
12      node-1                                  system.disk.total  5.867011e+09
13      node-4                                   system.disk.used  5.805428e+09
14      node-6                                  system.disk.total  5.283914e+09
15   adservice   runtime.java_lang_Threading_CurrentThreadCpuTime  5.011004e+09
16      node-3                             system.fs.inodes.total  4.860180e+09
17      node-2                             system.fs.inodes.total  4.860065e+09
18      node-1                             system.fs.inodes.total  4.860063e+09
19      node-3                              system.fs.inodes.free  4.858227e+09,                           trace_name_sample_up_to_50
0         trace.from_checkoutservice-0.duration_mean
1          trace.from_checkoutservice-0.duration_p95
2            trace.from_checkoutservice-0.error_rate
3             trace.from_checkoutservice-0.row_count
4         trace.from_checkoutservice-1.duration_mean
..                                               ...
45  trace.from_recommendationservice2-0.duration_p95
46    trace.from_recommendationservice2-0.error_rate
47     trace.from_recommendationservice2-0.row_count
48                     trace.from_root.duration_mean
49                      trace.from_root.duration_p95

[50 rows x 1 columns],                      cmdb_id                                   trace_name      p95
0                 frontend-0                         trace.self.row_count  1634.00
1                frontend2-0                         trace.self.row_count  1593.65
2                 frontend-2                         trace.self.row_count   899.80
3                 frontend-1                         trace.self.row_count   781.75
4                frontend2-0  trace.to_productcatalogservice2-0.row_count   744.85
5   productcatalogservice2-0             trace.from_frontend2-0.row_count   744.85
6                frontend2-0        trace.to_currencyservice2-0.row_count   496.65
7         currencyservice2-0             trace.from_frontend2-0.row_count   496.65
8    productcatalogservice-1              trace.from_frontend-0.row_count   251.40
9                 frontend-0   trace.to_productcatalogservice-1.row_count   251.40
10                frontend-0   trace.to_productcatalogservice-2.row_count   250.95
11   productcatalogservice-0              trace.from_frontend-0.row_count   250.95
12   productcatalogservice-2              trace.from_frontend-0.row_count   250.95
13                frontend-0   trace.to_productcatalogservice-0.row_count   250.95
14            cartservice2-0                         trace.self.row_count   170.00
15                frontend-0                    trace.from_root.row_count   164.10
16               frontend2-0                    trace.from_root.row_count   160.00
17         currencyservice-1              trace.from_frontend-0.row_count   159.55
18                frontend-0         trace.to_currencyservice-2.row_count   159.55
19         currencyservice-2              trace.from_frontend-0.row_count   159.55,   log_name_sample_up_to_50
0          log.error_count
1            log.row_count,                  timestamp     cmdb_id                                            message
0  2022-03-21 20:00:00 CST  frontend-0  severity: warning, message: failed to retrieve...
1  2022-03-21 20:00:00 CST  frontend-1  severity: warning, message: failed to retrieve...
2  2022-03-21 20:00:01 CST  frontend-2  severity: warning, message: failed to retrieve...
3  2022-03-21 20:00:03 CST  frontend-1  severity: warning, message: failed to retrieve...
4  2022-03-21 20:00:03 CST  frontend-2  severity: warning, message: failed to retrieve..., 5)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

# Reuse existing variables where possible: df_metric, tz (pytz timezone for Asia/Shanghai)
import pandas as pd
import numpy as np
import pytz

# ensure timezone variable
tz = pytz.timezone('Asia/Shanghai')

# Recompute global P95 per (cmdb_id, kpi_name) if not already available
# (Always compute to satisfy rule 9)
metric_p95 = df_metric.groupby(['cmdb_id','kpi_name'], dropna=False)['value'] \
                      .quantile(0.95) \
                      .reset_index(name='p95')

# Drop rows where p95 is NaN (no numeric data)
metric_p95 = metric_p95[metric_p95['p95'].notna()].copy()

# Prepare metric dataframe: add timezone-aware datetime column 'dt'
# Keep original 'timestamp' (epoch seconds). Create 'dt' in Asia/Shanghai.
df_metric['dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Define incident window in Asia/Shanghai (inclusive)
start = pd.Timestamp('2022-03-21 20:00:00', tz=tz)
end   = pd.Timestamp('2022-03-21 20:30:00', tz=tz)

# Filter metric rows within the incident window
df_window = df_metric[(df_metric['dt'] >= start) & (df_metric['dt'] <= end)].copy()

# Merge global p95 into the windowed data (per (cmdb_id,kpi_name))
df_window = df_window.merge(metric_p95, on=['cmdb_id','kpi_name'], how='left')

# Only keep rows where p95 exists (otherwise cannot evaluate anomaly)
df_window = df_window[df_window['p95'].notna()].copy()

# Flag anomalies where value > global P95 (strictly greater as requested)
df_window['is_anom'] = df_window['value'] > df_window['p95']

# Compute earliest anomaly timestamp per pair from the anomaly rows
anom_rows = df_window[df_window['is_anom']].copy()

if not anom_rows.empty:
    earliest = anom_rows.groupby(['cmdb_id','kpi_name'], as_index=False)['dt'].min().rename(columns={'dt':'earliest_anom_dt'})
else:
    # empty frame with expected columns
    earliest = pd.DataFrame(columns=['cmdb_id','kpi_name','earliest_anom_dt'])

# Aggregate window statistics per (cmdb_id, kpi_name, p95)
agg = df_window.groupby(['cmdb_id','kpi_name','p95'], as_index=False).agg(
    total_points = ('value','size'),
    anomaly_count = ('is_anom','sum'),
    max_value = ('value','max')
)

# Merge earliest anomaly timestamp
agg = agg.merge(earliest, on=['cmdb_id','kpi_name'], how='left')

# Keep only pairs with anomaly_count > 0
metric_anom_window = agg[agg['anomaly_count'] > 0].copy()

# Convert earliest_anom_dt to Asia/Shanghai formatted string (or NaN if missing)
metric_anom_window['earliest_anom_ts'] = metric_anom_window['earliest_anom_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

# Compute max_over_p95_ratio safely (avoid divide-by-zero)
def safe_ratio(max_v, p95_v):
    if pd.isna(p95_v) or p95_v == 0:
        return np.nan
    return max_v / p95_v

metric_anom_window['max_over_p95_ratio'] = metric_anom_window.apply(lambda r: safe_ratio(r['max_value'], r['p95']), axis=1)

# Select and order requested columns
metric_anom_window = metric_anom_window[[
    'cmdb_id','kpi_name','p95','anomaly_count','earliest_anom_ts','max_value','max_over_p95_ratio'
]]

# Sort by anomaly_count desc, then by max_over_p95_ratio desc
metric_anom_window = metric_anom_window.sort_values(by=['anomaly_count','max_over_p95_ratio'], ascending=[False, False]).reset_index(drop=True)

# Keep only top 20 rows for display as requested
metric_anom_window = metric_anom_window.head(20)

# Display the result (variable name as required)
metric_anom_window

```
Out[2]:
```
# Summarize the metric_anom_window (already computed) in plain English as a single concise string.
import pandas as pd

# Use existing metric_anom_window variable from kernel
df = metric_anom_window.copy()

n_rows = df.shape[0]
anom_counts = sorted(df['anomaly_count'].unique())
earliest_min = df['earliest_anom_ts'].dropna().min() if 'earliest_anom_ts' in df.columns else None
earliest_max = df['earliest_anom_ts'].dropna().max() if 'earliest_anom_ts' in df.columns else None

# Pattern counts
node5_net_recv_count = int(df['kpi_name'].str.contains('container.node-5.container_network_receive', na=False).sum())
retrans_count = int(df['kpi_name'].str.contains('system.net.tcp.retrans_segs', na=False).sum())
mesh_count = int(df['kpi_name'].str.contains(r'\bmesh\.', na=False).sum())

# Top examples by ratio (already sorted), include up to 3
top_examples = []
for _, r in df.head(3).iterrows():
    top_examples.append({
        'cmdb_id': r['cmdb_id'],
        'kpi_name': r['kpi_name'],
        'p95': float(r['p95']),
        'max_value': float(r['max_value']),
        'ratio': float(r['max_over_p95_ratio']),
        'earliest_anom_ts': r['earliest_anom_ts']
    })

summary = (
    f"Metric anomalies in window 2022-03-21 20:00:00 to 2022-03-21 20:30:00 (Asia/Shanghai):\n\n"
    f"- Returned rows: {n_rows} anomalous (cmdb_id, kpi_name) pairs (filtered to anomaly_count > 0).\n"
    f"- Observed anomaly_count values in these rows: {anom_counts} (most rows show anomaly_count=2).\n"
    f"- Earliest anomaly timestamps in the result range from {earliest_min} to {earliest_max}.\n\n"
    f"Key patterns in the top results:\n"
    f"- {node5_net_recv_count} entries involve 'container.node-5.container_network_receive...' (network receive metrics on node-5) showing elevated values over global P95.\n"
    f"- {retrans_count} entries are 'system.net.tcp.retrans_segs' (TCP retransmission) on nodes (notably node-2 and node-3).\n"
    f"- {mesh_count} entries are 'mesh.*' Istio mesh request/response metrics (several involving frontend and checkoutservice).\n\n"
    f"Top example anomalies (up to 3):\n"
)

for ex in top_examples:
    summary += (f"  • {ex['cmdb_id']} | {ex['kpi_name']} | p95={ex['p95']}, max={ex['max_value']}, "
                f"max/p95={ex['ratio']:.3f}, earliest={ex['earliest_anom_ts']}\n")

summary += (
    "\nInterpretation notes (observational only):\n"
    "- Many anomalies are concentrated on node-5 network receive metrics across multiple services, and TCP retransmission spikes appear on node-2/3.\n"
    "- Several Istio mesh metrics for frontend/checkout paths are also elevated. These are observed facts from the data; further investigation is required to determine causality.\n"
)

# Display the single summary string
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name          p95  anomaly_count         earliest_anom_ts    max_value  max_over_p95_ratio
0   checkoutservice-2  mesh.source.checkoutservice.cartservice.istio_...     0.012225              2  2022-03-21 20:05:00 CST   105.010350         8589.803681
1   checkoutservice-2  mesh.destination.frontend.checkoutservice.isti...     0.215950              2  2022-03-21 20:05:00 CST   115.368500          534.237092
2     checkoutservice                                       app.grpc.mrt     0.050465              2  2022-03-21 20:05:00 CST     7.604021          150.678168
3    paymentservice-1  container.node-5.container_network_receive_MB....     0.024996              2  2022-03-21 20:12:00 CST     0.550219           22.012205
4      emailservice-1  container.node-5.container_network_receive_MB....     0.029275              2  2022-03-21 20:12:00 CST     0.561360           19.175191
5      emailservice-2  container.node-5.container_network_receive_MB....     0.031426              2  2022-03-21 20:03:00 CST     0.556795           17.717886
6     emailservice2-0  container.node-5.container_network_receive_MB....     0.032304              2  2022-03-21 20:06:00 CST     0.555384           17.192183
7   shippingservice-2  container.node-5.container_network_receive_MB....     0.044042              2  2022-03-21 20:09:00 CST     0.569921           12.940320
8              node-4                                    system.io.svctm     0.045500              2  2022-03-21 20:13:00 CST     0.500000           10.989011
9   checkoutservice-0  container.node-5.container_network_receive_MB....     0.058021              2  2022-03-21 20:05:00 CST     0.586421           10.107094
10             node-2                        system.net.tcp.retrans_segs     0.962000              2  2022-03-21 20:12:00 CST     9.690000           10.072765
11        adservice-2  container.node-5.container_network_receive_MB....     0.077307              2  2022-03-21 20:05:00 CST     0.610030            7.890994
12             node-3                        system.net.tcp.retrans_segs     0.364000              2  2022-03-21 20:08:00 CST     2.280000            6.263736
13         frontend-0  mesh.destination.unknown.frontend.istio_reques...    11.137500              2  2022-03-21 20:05:00 CST    65.750000            5.903479
14         frontend-0  mesh.source.frontend.checkoutservice.istio_req...    11.137500              2  2022-03-21 20:05:00 CST    65.750000            5.903479
15         frontend-0  mesh.source.frontend.checkoutservice.istio_res...    86.625000              2  2022-03-21 20:05:00 CST   472.500000            5.454545
16         frontend-0  mesh.destination.unknown.frontend.istio_reques...   177.375000              2  2022-03-21 20:05:00 CST   967.500000            5.454545
17         frontend-0  mesh.destination.unknown.frontend.istio_reques...     0.275000              2  2022-03-21 20:05:00 CST     1.500000            5.454545
18         frontend-0  mesh.destination.unknown.frontend.istio_respon...  1223.750000              2  2022-03-21 20:05:00 CST  6675.000000            5.454545
19         frontend-0  mesh.source.frontend.checkoutservice.istio_req...   453.750000              2  2022-03-21 20:05:00 CST  2475.000000            5.454545```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# reuse existing df_trace and tz if present; otherwise set tz and ensure df_trace is loaded
tz = pytz.timezone('Asia/Shanghai')

# Ensure value is numeric
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

# 1) Compute global P95 per (cmdb_id, trace_name) using the entire trace series (do NOT filter first)
trace_p95 = df_trace.groupby(['cmdb_id','trace_name'], dropna=False)['value'] \
                    .quantile(0.95) \
                    .reset_index(name='p95')

# Drop groups without a numeric p95
trace_p95 = trace_p95[trace_p95['p95'].notna()].copy()

# 2) Prepare trace dataframe with timezone-aware datetime and filter the incident window
df_trace['dt'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

start = pd.Timestamp('2022-03-21 20:00:00', tz=tz)
end   = pd.Timestamp('2022-03-21 20:30:00', tz=tz)

df_window = df_trace[(df_trace['dt'] >= start) & (df_trace['dt'] <= end)].copy()

# 3) Merge global p95 into the windowed data
df_window = df_window.merge(trace_p95, on=['cmdb_id','trace_name'], how='left')

# keep only rows where p95 exists (otherwise cannot evaluate anomaly)
df_window = df_window[df_window['p95'].notna()].copy()

# 4) Flag anomalies where value > global P95
df_window['is_anom'] = df_window['value'] > df_window['p95']

# 5) Earliest anomaly timestamp per pair (from anomaly rows)
anom_rows = df_window[df_window['is_anom']].copy()
if not anom_rows.empty:
    earliest = anom_rows.groupby(['cmdb_id','trace_name'], as_index=False)['dt'].min().rename(columns={'dt':'earliest_anom_dt'})
else:
    earliest = pd.DataFrame(columns=['cmdb_id','trace_name','earliest_anom_dt'])

# 6) Aggregate window stats per (cmdb_id, trace_name, p95)
agg = df_window.groupby(['cmdb_id','trace_name','p95'], as_index=False).agg(
    total_points = ('value','size'),
    anomaly_count = ('is_anom','sum'),
    max_value = ('value','max')
)

# merge earliest anomaly timestamp
agg = agg.merge(earliest, on=['cmdb_id','trace_name'], how='left')

# Keep only pairs with anomaly_count > 0
trace_anom_window = agg[agg['anomaly_count'] > 0].copy()

# Format earliest anomaly timestamp and compute ratio
trace_anom_window['earliest_anom_ts'] = trace_anom_window['earliest_anom_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

def safe_ratio(max_v, p95_v):
    if pd.isna(p95_v) or p95_v == 0:
        return np.nan
    return max_v / p95_v

trace_anom_window['max_over_p95_ratio'] = trace_anom_window.apply(lambda r: safe_ratio(r['max_value'], r['p95']), axis=1)

# Select requested columns and sort as specified
trace_anom_window = trace_anom_window[[
    'cmdb_id','trace_name','p95','anomaly_count','earliest_anom_ts','max_value','max_over_p95_ratio'
]]

trace_anom_window = trace_anom_window.sort_values(by=['anomaly_count','max_over_p95_ratio'], ascending=[False, False]).reset_index(drop=True)

# Keep top 20 rows for compact display
trace_anom_window = trace_anom_window.head(20)

# Display the result (variable as required)
trace_anom_window

```
Out[3]:
```
Summary of trace anomalies in the incident window (2022-03-21 20:00–20:30 CST):

- I computed global P95 per (cmdb_id, trace_name) using the full trace series and flagged window points with value > that global P95. The result returned 20 (cmdb_id, trace_name) pairs with anomalies (all shown are anomaly_count = 2).
- The most severe spikes are latency/duration metrics on the frontend ⇄ checkoutservice path:
  - checkoutservice-2 | trace.from_frontend-2.duration_p95 — p95=0.155967, max=107.998323, max/p95 ≈ 692.44, earliest anomalous point 2022-03-21 20:03:00 CST.
  - frontend-2 | trace.to_checkoutservice-2.duration_p95 — same spike mirrored on the frontend side (p95=0.155967, max=107.998323, ratio ≈ 692.44, earliest 20:03).
  - Similar large ratios appear for duration_mean variants between frontend and checkoutservice (e.g., ratio ≈ 365.7).
- Other notable anomalies include:
  - checkoutservice-2 trace.from_frontend-0 and frontend-0 → checkoutservice-2 duration metrics (large ratios, earliest ~20:04).
  - Smaller but clear spikes for cartservice-1 and various frontend self/from_root duration metrics (ratios from ~6× up to ~25× depending on tiny p95 baselines).
- Timing: earliest anomalous timestamps cluster around 20:03–20:05, with some later anomalies near 20:10–20:22 for other trace flows.
- Interpretation: the trace data shows pronounced latency spikes concentrated on the frontend → checkoutservice call paths (especially involving checkoutservice-2 and frontend-2) at ~20:03–20:05, suggesting a service-level latency incident affecting checkout-related traces. Other services show smaller relative spikes.
- Recommended next steps: inspect corresponding traces in detail for these request ids, check related metrics (mesh request/response counts, network I/O, retransmissions) and logs for frontend/checkoutservice around 20:03–20:05 to determine root cause (e.g., backend latency, network issues, or resource saturation).

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                        trace_name       p95  anomaly_count         earliest_anom_ts   max_value  max_over_p95_ratio
0         checkoutservice-2                trace.from_frontend-2.duration_p95  0.155967              2  2022-03-21 20:03:00 CST  107.998323          692.442977
1                frontend-2           trace.to_checkoutservice-2.duration_p95  0.155967              2  2022-03-21 20:03:00 CST  107.998323          692.442977
2         checkoutservice-2               trace.from_frontend-2.duration_mean  0.155491              2  2022-03-21 20:03:00 CST   56.859457          365.676837
3                frontend-2          trace.to_checkoutservice-2.duration_mean  0.155491              2  2022-03-21 20:03:00 CST   56.859457          365.676837
4         checkoutservice-2               trace.from_frontend-0.duration_mean  0.053924              2  2022-03-21 20:04:00 CST   14.980622          277.810948
5                frontend-0          trace.to_checkoutservice-2.duration_mean  0.053924              2  2022-03-21 20:04:00 CST   14.980622          277.810948
6         checkoutservice-2                trace.from_frontend-0.duration_p95  0.079234              2  2022-03-21 20:04:00 CST   20.000441          252.423732
7                frontend-0           trace.to_checkoutservice-2.duration_p95  0.079234              2  2022-03-21 20:04:00 CST   20.000441          252.423732
8             cartservice-1                trace.from_frontend-0.duration_p95  0.000027              2  2022-03-21 20:22:00 CST    0.000700           25.454545
9                frontend-0               trace.to_cartservice-1.duration_p95  0.000027              2  2022-03-21 20:22:00 CST    0.000700           25.454545
10               frontend-1                          trace.self.duration_mean  0.004083              2  2022-03-21 20:03:00 CST    0.042728           10.465751
11        checkoutservice-2           trace.to_currencyservice-0.duration_p95  0.000132              2  2022-03-21 20:10:00 CST    0.001354           10.257576
12        currencyservice-0         trace.from_checkoutservice-2.duration_p95  0.000132              2  2022-03-21 20:10:00 CST    0.001354           10.257576
13               frontend-1                     trace.from_root.duration_mean  0.042297              2  2022-03-21 20:03:00 CST    0.423520           10.012932
14               frontend-0                     trace.from_root.duration_mean  0.143857              2  2022-03-21 20:04:00 CST    1.086015            7.549283
15               frontend-0                          trace.self.duration_mean  0.014423              2  2022-03-21 20:04:00 CST    0.107317            7.440891
16        checkoutservice-2                      trace.from_root.duration_p95  0.003872              2  2022-03-21 20:05:00 CST    0.026067            6.731484
17        checkoutservice-1           trace.to_currencyservice-0.duration_p95  0.000151              2  2022-03-21 20:07:00 CST    0.001014            6.721909
18        currencyservice-0         trace.from_checkoutservice-1.duration_p95  0.000151              2  2022-03-21 20:07:00 CST    0.001014            6.721909
19  productcatalogservice-0  trace.from_recommendationservice2-0.duration_p95  0.000038              2  2022-03-21 20:03:00 CST    0.000252            6.693227```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# reuse existing dataframes: df_err (error_logs.csv) and df_log (log.csv)
# Ensure timezone
tz = pytz.timezone('Asia/Shanghai')

# Target components and time window (inclusive)
targets = ['frontend-0','frontend-1','frontend-2',
           'checkoutservice-0','checkoutservice-1','checkoutservice-2']
start = pd.Timestamp('2022-03-21 20:00:00', tz=tz)
end   = pd.Timestamp('2022-03-21 20:06:00', tz=tz)

# Prepare df_err: ensure timestamp column as datetime in Asia/Shanghai
df_err['dt'] = pd.to_datetime(df_err['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) error_logs.csv: filter by cmdb_id and time window
err_window = df_err[(df_err['cmdb_id'].isin(targets)) & (df_err['dt'] >= start) & (df_err['dt'] <= end)].copy()

# Aggregate per cmdb_id: count, earliest timestamp, up to 10 sample messages
def sample_messages(series, n=10):
    vals = series.dropna().astype(str).unique()
    return " || ".join(vals[:n])

if not err_window.empty:
    errorlog_window_summary = err_window.groupby('cmdb_id', as_index=False).agg(
        row_count = ('message','size'),
        earliest_dt = ('dt','min'),
        sample_messages = ('message', lambda s: sample_messages(s, n=10))
    )
    errorlog_window_summary['earliest_ts'] = errorlog_window_summary['earliest_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    errorlog_window_summary = errorlog_window_summary[['cmdb_id','row_count','earliest_ts','sample_messages']]
else:
    # empty frame with expected columns
    errorlog_window_summary = pd.DataFrame(columns=['cmdb_id','row_count','earliest_ts','sample_messages'])

# Limit to at most 20 rows (there will be <=6)
errorlog_window_summary = errorlog_window_summary.head(20)

# 2) log.csv: prepare dt and ensure numeric value
df_log['dt'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')

log_window = df_log[(df_log['cmdb_id'].isin(targets)) & (df_log['dt'] >= start) & (df_log['dt'] <= end)].copy()

if not log_window.empty and 'log_name' in log_window.columns:
    log_window_agg = log_window.groupby(['cmdb_id','log_name'], as_index=False).agg(
        points_in_window = ('value','size'),
        sum_value = ('value','sum')
    )
    # Round sum_value for compactness
    log_window_agg['sum_value'] = log_window_agg['sum_value'].round(6)
else:
    log_window_agg = pd.DataFrame(columns=['cmdb_id','log_name','points_in_window','sum_value'])

# Limit to top 20 rows for display
log_window_agg = log_window_agg.head(20)

# Display the two compact tables (as required by rule 2, return variables)
errorlog_window_summary, log_window_agg

```
Out[4]:
```
Summary (window 2022-03-21 20:00:00–20:06:00 CST) — focused on frontends and checkoutservice:

What I found
- error_logs (filtered to frontend-0/1/2 and checkoutservice-0/1/2)
  - frontend-0: 184 rows, earliest 2022-03-21 20:00:00 CST. Sample messages show repeated "severity: warning, message: failed to retrieve..." (and similar warnings).
  - frontend-1: 87 rows, earliest 2022-03-21 20:00:00 CST. Same sample warning messages.
  - frontend-2: 102 rows, earliest 2022-03-21 20:00:01 CST. Same sample warning messages.
  - (checkoutservice-* produced no rows in error_logs in this filtered window.)
- log.csv aggregates (per cmdb_id and log_name in the same window)
  - Each (cmdb_id, log_name) has 7 points in-window (one per minute across 7 minutes).
  - checkoutservice-0: log.row_count sum = 460.0, log.error_count sum = 0.0
  - checkoutservice-1: log.row_count sum = 453.0, log.error_count sum = 0.0
  - checkoutservice-2: log.row_count sum = 409.0, log.error_count sum = 0.0
  - frontend-0: log.row_count sum = 16168.0, log.error_count sum = 216.0
  - frontend-1: log.row_count sum = 7541.0,  log.error_count sum = 103.0
  - frontend-2: log.row_count sum = 9138.0,  log.error_count sum = 119.0

Interpretation / correlation with traces & metrics
- Frontend components show many warning/error log entries starting right at the onset (20:00:00–20:00:01) and substantial summed error_count values over the window (frontend-0: 216, frontend-1: 103, frontend-2: 119). Sample messages indicate failed retrievals from backends.
- Checkout services do not show error_count in logs during this window (error_count sums = 0), though they handled substantial row_count (requests).
- This corroborates the trace/metric signals: traces showed large latency spikes on frontend ⇄ checkoutservice paths (~20:03–20:05), and metrics showed network/mesh and retransmission signals. The logs suggest frontends experienced failing/slow responses when trying to retrieve data from downstream (checkoutservice or other backends), rather than checkoutservice emitting many errors itself.

Suggested next steps
- Inspect full frontend log entries (expand the "failed to retrieve..." messages) around 20:00–20:06 to find timeouts, stack traces, or upstream error details.
- Correlate specific request IDs from frontend warnings with the trace spans showing large durations (frontend ⇄ checkoutservice at ~20:03) to identify whether backend processing or network issues caused the latency/failures.
- Check network-related metrics already flagged (node-5 network receive, TCP retransmissions) for the involved nodes to determine if network packet loss/retransmission contributed.

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id  row_count              earliest_ts                                    sample_messages
0  frontend-0        184  2022-03-21 20:00:00 CST  severity: warning, message: failed to retrieve...
1  frontend-1         87  2022-03-21 20:00:00 CST  severity: warning, message: failed to retrieve...
2  frontend-2        102  2022-03-21 20:00:01 CST  severity: warning, message: failed to retrieve...,               cmdb_id         log_name  points_in_window  sum_value
0   checkoutservice-0  log.error_count                 7        0.0
1   checkoutservice-0    log.row_count                 7      460.0
2   checkoutservice-1  log.error_count                 7        0.0
3   checkoutservice-1    log.row_count                 7      453.0
4   checkoutservice-2  log.error_count                 7        0.0
5   checkoutservice-2    log.row_count                 7      409.0
6          frontend-0  log.error_count                 7      216.0
7          frontend-0    log.row_count                 7    16168.0
8          frontend-1  log.error_count                 7      103.0
9          frontend-1    log.row_count                 7     7541.0
10         frontend-2  log.error_count                 7      119.0
11         frontend-2    log.row_count                 7     9138.0)```
```

In [ ]:
# In[5]:

# Reuse existing df_metric from kernel and compute counts for kpi_name containing 'container.node-5'
import pandas as pd

mask_node5 = df_metric['kpi_name'].astype(str).str.contains('container.node-5', na=False)
df_node5 = df_metric[mask_node5].copy()

if not df_node5.empty:
    node5_cmdbs = df_node5.groupby('cmdb_id').size().reset_index()
    node5_cmdbs.columns = ['cmdb_id', 'count_of_node5_kpi_rows']
    node5_cmdbs = node5_cmdbs.sort_values('count_of_node5_kpi_rows', ascending=False).reset_index(drop=True)
else:
    node5_cmdbs = pd.DataFrame(columns=['cmdb_id', 'count_of_node5_kpi_rows'])

# Check presence of specific components
checkoutservice2_in_node5 = 'checkoutservice-2' in node5_cmdbs['cmdb_id'].values
frontend2_in_node5 = 'frontend-2' in node5_cmdbs['cmdb_id'].values

# Display up to 50 rows and the booleans
node5_cmdbs.head(50), checkoutservice2_in_node5, frontend2_in_node5

```
Out[5]:
```
# Summarize node5_cmdbs presence and top counts in plain English using existing kernel variables.
# Reuse node5_cmdbs, checkoutservice2_in_node5, frontend2_in_node5 from the kernel.

top_n = 5
total_cmdbs = int(node5_cmdbs.shape[0])
top_examples = node5_cmdbs.head(top_n).copy()

# Build a concise string for top examples
examples_str = ', '.join([f"{row['cmdb_id']}:{int(row['count_of_node5_kpi_rows'])}" 
                          for _, row in top_examples.iterrows()])

summary = (
    f"Found {total_cmdbs} distinct cmdb_id entries that have KPIs containing 'container.node-5'. "
    f"Top {min(top_n, total_cmdbs)} examples: {examples_str}. "
    f"Presence checks: checkoutservice-2 = {bool(checkoutservice2_in_node5)}, frontend-2 = {bool(frontend2_in_node5)}."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id  count_of_node5_kpi_rows
0         adservice-0                     1920
1         adservice-1                     1920
2         adservice-2                     1920
3   checkoutservice-0                     1920
4       cartservice-2                     1920
..                ...                      ...
30  shippingservice-0                     1920
31      cartservice-0                     1800
32      cartservice-1                     1800
33      redis-cart2-0                      240
34       redis-cart-0                      240

[35 rows x 2 columns], True, True)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

# Filter metric_anom_window for kpi_name containing 'container.node-5'
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing metric_anom_window if present; otherwise recompute (fallback)
if 'metric_anom_window' not in globals():
    # Recompute global P95 and metric_anom_window following previous logic (uses full metric series for thresholds)
    df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
    metric_p95 = df_metric.groupby(['cmdb_id','kpi_name'], dropna=False)['value'].quantile(0.95).reset_index(name='p95')
    metric_p95 = metric_p95[metric_p95['p95'].notna()].copy()
    df_metric['dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    start = pd.Timestamp('2022-03-21 20:00:00', tz=tz)
    end   = pd.Timestamp('2022-03-21 20:30:00', tz=tz)
    df_window = df_metric[(df_metric['dt'] >= start) & (df_metric['dt'] <= end)].copy()
    df_window = df_window.merge(metric_p95, on=['cmdb_id','kpi_name'], how='left')
    df_window = df_window[df_window['p95'].notna()].copy()
    df_window['is_anom'] = df_window['value'] > df_window['p95']
    anom_rows = df_window[df_window['is_anom']].copy()
    if not anom_rows.empty:
        earliest = anom_rows.groupby(['cmdb_id','kpi_name'], as_index=False)['dt'].min().rename(columns={'dt':'earliest_anom_dt'})
    else:
        earliest = pd.DataFrame(columns=['cmdb_id','kpi_name','earliest_anom_dt'])
    agg = df_window.groupby(['cmdb_id','kpi_name','p95'], as_index=False).agg(
        total_points = ('value','size'),
        anomaly_count = ('is_anom','sum'),
        max_value = ('value','max')
    )
    agg = agg.merge(earliest, on=['cmdb_id','kpi_name'], how='left')
    metric_anom_window = agg[agg['anomaly_count'] > 0].copy()
    metric_anom_window['earliest_anom_ts'] = metric_anom_window['earliest_anom_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    metric_anom_window['max_over_p95_ratio'] = metric_anom_window.apply(lambda r: (r['max_value']/r['p95']) if (pd.notna(r['p95']) and r['p95']!=0) else pd.NA, axis=1)
    metric_anom_window = metric_anom_window[[
        'cmdb_id','kpi_name','p95','anomaly_count','earliest_anom_ts','max_value','max_over_p95_ratio'
    ]]

# Now filter for 'container.node-5' in kpi_name
node5_anom_details = metric_anom_window[metric_anom_window['kpi_name'].astype(str).str.contains('container.node-5', na=False)].copy()

# Parse earliest_anom_ts (which is formatted as 'YYYY-MM-DD HH:MM:SS CST') into timezone-aware datetime for sorting.
# Remove timezone label and localize to Asia/Shanghai to ensure consistent ordering.
if 'earliest_anom_ts' in node5_anom_details.columns and not node5_anom_details['earliest_anom_ts'].isna().all():
    # Remove trailing timezone label (e.g., ' CST') if present, then parse and localize
    cleaned = node5_anom_details['earliest_anom_ts'].astype(str).str.replace(r'\s+[A-Z]+$', '', regex=True)
    node5_anom_details['earliest_anom_dt'] = pd.to_datetime(cleaned, format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.tz_localize(tz)
else:
    node5_anom_details['earliest_anom_dt'] = pd.NaT

# Ensure numeric ratio column
node5_anom_details['max_over_p95_ratio'] = pd.to_numeric(node5_anom_details['max_over_p95_ratio'], errors='coerce')

# Sort by earliest_anom_ts ascending (earliest first), then by max_over_p95_ratio descending
node5_anom_details = node5_anom_details.sort_values(by=['earliest_anom_dt','max_over_p95_ratio'], ascending=[True, False])

# Keep requested columns and up to 20 rows
node5_anom_details = node5_anom_details[['cmdb_id','kpi_name','p95','anomaly_count','earliest_anom_ts','max_value','max_over_p95_ratio']].head(20).reset_index(drop=True)

# Display the result
node5_anom_details

```
Out[6]:
```
# Summarize node5_anom_details in plain English using existing kernel variable
# Reuse node5_anom_details from kernel
rows = node5_anom_details.to_dict(orient='records')

if rows:
    earliest = rows[0]['earliest_anom_ts']
    latest = rows[-1]['earliest_anom_ts']
    services = [r['cmdb_id'] for r in rows]
    top_ratio_row = max(rows, key=lambda r: (r['max_over_p95_ratio'] if pd.notna(r['max_over_p95_ratio']) else -1))
    summary = (
        "Summary of node-5 related metric anomalies (container.node-5.*):\n\n"
        f"- Found {len(rows)} (cmdb_id, kpi_name) anomaly rows (all with anomaly_count=2) within the incident window.\n"
        f"- Earliest anomaly observed: {earliest}; latest shown: {latest} (Asia/Shanghai).\n"
        "- Affected components (in order shown): " + ", ".join(services) + ".\n"
        f"- Highest severity (max_value / p95) observed on {top_ratio_row['cmdb_id']} for metric '{top_ratio_row['kpi_name']}' "
        f"with ratio ≈ {top_ratio_row['max_over_p95_ratio']:.3f} (p95={top_ratio_row['p95']}, max={top_ratio_row['max_value']}).\n\n"
        "Interpretation:\n"
        "- Multiple services show spikes on 'container.node-5.container_network_receive...' (network receive metrics tied to node-5), "
        "with earliest spikes around 20:03–20:06 and some later at ~20:09–20:12. This pattern suggests a node-5 network receive anomaly "
        "affecting many containers/services rather than a single service failure.\n"
        "- Correlate these node-5 network receive spikes with the earlier trace/metric indications (frontend ⇄ checkoutservice latency and TCP retransmissions) "
        "to investigate node/network-level packet loss or congestion as a potential root cause.\n"
    )
else:
    summary = "No node-5 related anomalies found in metric_anom_window."

summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name       p95  anomaly_count         earliest_anom_ts  max_value  max_over_p95_ratio
0     emailservice-2  container.node-5.container_network_receive_MB....  0.031426              2  2022-03-21 20:03:00 CST   0.556795           17.717886
1  checkoutservice-0  container.node-5.container_network_receive_MB....  0.058021              2  2022-03-21 20:05:00 CST   0.586421           10.107094
2        adservice-2  container.node-5.container_network_receive_MB....  0.077307              2  2022-03-21 20:05:00 CST   0.610030            7.890994
3    emailservice2-0  container.node-5.container_network_receive_MB....  0.032304              2  2022-03-21 20:06:00 CST   0.555384           17.192183
4  shippingservice-2  container.node-5.container_network_receive_MB....  0.044042              2  2022-03-21 20:09:00 CST   0.569921           12.940320
5   paymentservice-1  container.node-5.container_network_receive_MB....  0.024996              2  2022-03-21 20:12:00 CST   0.550219           22.012205
6     emailservice-1  container.node-5.container_network_receive_MB....  0.029275              2  2022-03-21 20:12:00 CST   0.561360           19.175191```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure df_metric exists in kernel (it was loaded earlier); otherwise load it
# (We assume df_metric is present from prior steps per session state.)
if 'df_metric' not in globals():
    df_metric = pd.read_csv('metric.csv')
    df_metric['value'] = pd.to_numeric(df_metric.get('value'), errors='coerce')

# Recompute metric_anom_window if not present
if 'metric_anom_window' not in globals():
    # 1) Compute global P95 per (cmdb_id,kpi_name) using full series
    df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
    metric_p95 = df_metric.groupby(['cmdb_id','kpi_name'], dropna=False)['value'].quantile(0.95).reset_index(name='p95')
    metric_p95 = metric_p95[metric_p95['p95'].notna()].copy()

    # 2) Prepare windowed data
    df_metric['dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    start = pd.Timestamp('2022-03-21 20:00:00', tz=tz)
    end   = pd.Timestamp('2022-03-21 20:30:00', tz=tz)
    df_window = df_metric[(df_metric['dt'] >= start) & (df_metric['dt'] <= end)].copy()

    # 3) Merge p95 and flag anomalies
    df_window = df_window.merge(metric_p95, on=['cmdb_id','kpi_name'], how='left')
    df_window = df_window[df_window['p95'].notna()].copy()
    df_window['is_anom'] = df_window['value'] > df_window['p95']

    # 4) Earliest anomaly timestamp per series
    anom_rows = df_window[df_window['is_anom']].copy()
    if not anom_rows.empty:
        earliest = anom_rows.groupby(['cmdb_id','kpi_name'], as_index=False)['dt'].min().rename(columns={'dt':'earliest_anom_dt'})
    else:
        earliest = pd.DataFrame(columns=['cmdb_id','kpi_name','earliest_anom_dt'])

    # 5) Aggregate per series
    agg = df_window.groupby(['cmdb_id','kpi_name','p95'], as_index=False).agg(
        total_points = ('value','size'),
        anomaly_count = ('is_anom','sum'),
        max_value = ('value','max')
    )
    agg = agg.merge(earliest, on=['cmdb_id','kpi_name'], how='left')
    agg = agg[agg['anomaly_count'] > 0].copy()
    agg['earliest_anom_ts'] = agg['earliest_anom_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    agg['max_over_p95_ratio'] = agg.apply(lambda r: (r['max_value']/r['p95']) if (pd.notna(r['p95']) and r['p95']!=0) else np.nan, axis=1)

    metric_anom_window = agg[[
        'cmdb_id','kpi_name','p95','anomaly_count','earliest_anom_ts','max_value','max_over_p95_ratio'
    ]].reset_index(drop=True)

# Now aggregate anomalies by cmdb_id to produce severity ranking
# Ensure numeric ratio and parse earliest timestamp for accurate min computation
df_rank = metric_anom_window.copy()

# Parse earliest_anom_ts into timezone-aware datetimes for min
if 'earliest_anom_ts' in df_rank.columns:
    cleaned = df_rank['earliest_anom_ts'].astype(str).replace('nan', pd.NA)
    cleaned = cleaned.str.replace(r'\s+[A-Z]+$', '', regex=True)  # strip tz abbrev
    df_rank['earliest_anom_dt'] = pd.to_datetime(cleaned, format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.tz_localize(tz)
else:
    df_rank['earliest_anom_dt'] = pd.NaT

df_rank['max_over_p95_ratio'] = pd.to_numeric(df_rank['max_over_p95_ratio'], errors='coerce')
df_rank['anomaly_count'] = pd.to_numeric(df_rank['anomaly_count'], errors='coerce')

# Group by cmdb_id
comp_agg = df_rank.groupby('cmdb_id', as_index=False).agg(
    total_anomaly_count = ('anomaly_count','sum'),
    max_over_p95_ratio = ('max_over_p95_ratio','max'),
    earliest_anom_dt = ('earliest_anom_dt','min')
)

# Format earliest timestamp back to string in Asia/Shanghai (or NaN)
comp_agg['earliest_anom_ts'] = comp_agg['earliest_anom_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

# Boolean is_node if cmdb_id starts with 'node-'
comp_agg['is_node'] = comp_agg['cmdb_id'].astype(str).str.startswith('node-')

# Sort by max_over_p95_ratio desc, ties by total_anomaly_count desc
comp_anom_ranking = comp_agg.sort_values(by=['max_over_p95_ratio','total_anomaly_count'], ascending=[False, False]).reset_index(drop=True)

# Keep requested columns and top 20 rows for compact display
comp_anom_ranking = comp_anom_ranking[['cmdb_id','total_anomaly_count','max_over_p95_ratio','earliest_anom_ts','is_node']].head(20)

# Display result variable
comp_anom_ranking

```
Out[7]:
```
Summary of per-component severity ranking (top results):

- Highest severity (by max_over_p95_ratio):
  - checkoutservice-2 — total_anomaly_count = 4, max_over_p95_ratio ≈ 8589.80, earliest anomaly at 2022-03-21 20:05:00 CST.
  - checkoutservice (service-level) — total_anomaly_count = 2, max_over_p95_ratio ≈ 150.68, earliest 2022-03-21 20:05:00 CST.
  - paymentservice-1 — total_anomaly_count = 2, max_over_p95_ratio ≈ 22.01, earliest 2022-03-21 20:12:00 CST.
  - emailservice-1 / emailservice-2 / emailservice2-0 — each 2 anomalies, ratios ≈ 19.18, 17.72, 17.19 respectively, earliest around 20:03–20:12.
  - shippingservice-2 and checkoutservice-0 — each 2 anomalies, ratios ≈ 12.94 and 10.11, earliest ~20:05–20:09.
- Node-level entries present among the top ranks:
  - node-4, node-2, node-3 all appear with anomalies (is_node = True), each with anomaly_count = 2 and max_over_p95_ratio ≈ 10.99, 10.07, 6.26 respectively.
- Notable volume signal:
  - frontend-0 has the largest total_anomaly_count shown here (14 anomalies) though its max_over_p95_ratio is more modest (~5.90), earliest at 20:05.

Interpretation / recommended focus:
- checkoutservice-2 is the single most extreme outlier by ratio and should be investigated immediately (earliest flagged at 20:05).
- frontend-0 shows many anomalous KPI series (high count) around the same onset time (20:05) and is likely heavily impacted.
- Node-level anomalies (node-4, node-2, node-3) are present — investigate node health and network I/O on those nodes as they may explain cross-service issues.
- Next steps: correlate checkoutservice-2 and frontend-0 anomalies with traces, logs, and node/network metrics (retransmissions, node-5 network receive spikes) around 20:03–20:12 to identify root cause.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  total_anomaly_count  max_over_p95_ratio         earliest_anom_ts  is_node
0   checkoutservice-2                    4         8589.803681  2022-03-21 20:05:00 CST    False
1     checkoutservice                    2          150.678168  2022-03-21 20:05:00 CST    False
2    paymentservice-1                    2           22.012205  2022-03-21 20:12:00 CST    False
3      emailservice-1                    2           19.175191  2022-03-21 20:12:00 CST    False
4      emailservice-2                    2           17.717886  2022-03-21 20:03:00 CST    False
5     emailservice2-0                    2           17.192183  2022-03-21 20:06:00 CST    False
6   shippingservice-2                    2           12.940320  2022-03-21 20:09:00 CST    False
7              node-4                    2           10.989011  2022-03-21 20:13:00 CST     True
8   checkoutservice-0                    2           10.107094  2022-03-21 20:05:00 CST    False
9              node-2                    2           10.072765  2022-03-21 20:12:00 CST     True
10        adservice-2                    2            7.890994  2022-03-21 20:05:00 CST    False
11             node-3                    2            6.263736  2022-03-21 20:08:00 CST     True
12         frontend-0                   14            5.903479  2022-03-21 20:05:00 CST    False```
```